In [1]:
import os, re, json, pandas as pd
from scipy.stats import randint
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

# =======================
# Localizar o arquivo CSV
# =======================
paths = [
    "./datasets/phishing_transformed.csv","../datasets/phishing_transformed.csv",
    "./datasets/phishing.csv","../datasets/phishing.csv","../../datasets/phishing.csv",
    "phishing_transformed.csv","phishing.csv"
]
csv = next((p for p in paths if os.path.exists(p)), None)
if not csv:
    raise FileNotFoundError("Coloque phishing_transformed.csv ou phishing.csv em ./datasets/")

try:
    df = pd.read_csv(csv, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv, encoding="latin1")

# =======================
# Definir alvo (y)
# =======================
if "Email Type_Phishing Email" in df.columns:
    y = df["Email Type_Phishing Email"].astype(int)
elif "Email Type" in df.columns:
    y = df["Email Type"].astype(str).str.lower().str.contains("phishing").astype(int)
else:
    raise ValueError("Alvo de phishing não encontrado (colunas 'Email Type_Phishing Email' ou 'Email Type').")

# =======================
# Limpeza e preparação de texto (se existir)
# =======================
def clean(s):
    s = str(s).lower()
    s = re.sub(r'[^a-zA-Z0-9áéíóúãõâêôçÁÉÍÓÚÃÕÂÊÔÇ\s]', ' ', s)
    s = re.sub(r'\b\d+\b', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

has_text = "Email Text" in df.columns
if has_text:
    df["Email Text"] = df["Email Text"].apply(clean)

# =======================
# Features (X) e ColumnTransformer robusto
# =======================
drop_cols = [c for c in ["Email Type","Email Type_Phishing Email"] if c in df.columns]
X = df.drop(columns=drop_cols)

num_cols = X.select_dtypes(include=["number"]).columns.tolist()

transformers = []
if has_text:
    # Vetorização de texto (BoW)
    transformers.append(("text", CountVectorizer(max_features=4000, min_df=5, max_df=0.8), "Email Text"))
    # Se houver numéricas, padroniza também
    if len(num_cols) > 0:
        transformers.append(("num", StandardScaler(with_mean=False), num_cols))
else:
    # Sem texto: só numéricas
    if len(num_cols) == 0:
        raise ValueError("Nenhuma coluna de texto ('Email Text') ou numérica disponível para treinamento.")
    transformers.append(("num", StandardScaler(with_mean=False), num_cols))

prep = ColumnTransformer(transformers, remainder="drop")

# =======================
# Avaliação por múltiplas seeds (CV 5-fold)
# =======================
SEEDS = list(range(20))
cv_results_per_seed = []

for seed in SEEDS:
    pipeline = Pipeline([
        ("prep", prep),
        ("clf", RandomForestClassifier(random_state=seed))
    ])

    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    scores = cross_val_score(pipeline, X, y, scoring="accuracy", cv=cv, n_jobs=-1)
    mean_acc, std_acc = scores.mean(), scores.std()
    print(f"[Seed:{seed:02d}] KFold-5 acc: {mean_acc:.4f} ± {std_acc:.4f}")

    cv_results_per_seed.append({
        "seed": seed,
        "cv_folds": 5,
        "accuracy_mean": mean_acc,
        "accuracy_std": std_acc
    })

# =======================
# RandomizedSearchCV (usa a última seed do loop)
# =======================
param_dist = {
    "clf__n_estimators": randint(150, 600),
    "clf__max_depth": [None, 6, 10, 16, 24],
    "clf__min_samples_split": randint(2, 12),
    "clf__min_samples_leaf": randint(1, 8),
    "clf__max_features": ["sqrt", "log2", 0.5],
}

pipeline = Pipeline([
    ("prep", prep),
    ("clf", RandomForestClassifier(random_state=seed))
])
cv = StratifiedKFold(5, shuffle=True, random_state=seed)

rs = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    random_state=seed,
    verbose=1
)

rs.fit(X, y)
print(f"\n✅ [RandomSearch] best_acc: {rs.best_score_:.4f}")
print(f"📌 best_params: {rs.best_params_}")

# =======================
# Salvar resultados em CSV/JSON
# =======================
os.makedirs("results", exist_ok=True)

# 1) Resultados por seed (K-Fold)
df_seeds = pd.DataFrame(cv_results_per_seed)
csv_seeds_path = "results/phishing_rf_cv5_seeds.csv"
df_seeds.to_csv(csv_seeds_path, index=False, encoding="utf-8")
print(f"💾 CSV salvo: {csv_seeds_path} (linhas: {len(df_seeds)})")

# 2) Tabela completa do RandomizedSearchCV (cv_results_)
df_rs = pd.DataFrame(rs.cv_results_)
csv_rs_path = "results/phishing_rf_randomsearch_cv5.csv"
df_rs.to_csv(csv_rs_path, index=False, encoding="utf-8")
print(f"💾 CSV salvo: {csv_rs_path} (linhas: {len(df_rs)})")

# 3) Best params em JSON
best_params_path = "results/phishing_rf_best_params.json"
with open(best_params_path, "w", encoding="utf-8") as f:
    json.dump(rs.best_params_, f, ensure_ascii=False, indent=2)
print(f"💾 JSON salvo: {best_params_path}")


[Seed:00] KFold-5 acc: 0.9557 ± 0.0016
[Seed:01] KFold-5 acc: 0.9538 ± 0.0023
[Seed:02] KFold-5 acc: 0.9553 ± 0.0032
[Seed:03] KFold-5 acc: 0.9550 ± 0.0040
[Seed:04] KFold-5 acc: 0.9572 ± 0.0023
[Seed:05] KFold-5 acc: 0.9555 ± 0.0042
[Seed:06] KFold-5 acc: 0.9557 ± 0.0013
[Seed:07] KFold-5 acc: 0.9557 ± 0.0033
[Seed:08] KFold-5 acc: 0.9560 ± 0.0028
[Seed:09] KFold-5 acc: 0.9568 ± 0.0033
[Seed:10] KFold-5 acc: 0.9553 ± 0.0016
[Seed:11] KFold-5 acc: 0.9558 ± 0.0027
[Seed:12] KFold-5 acc: 0.9562 ± 0.0036
[Seed:13] KFold-5 acc: 0.9562 ± 0.0029
[Seed:14] KFold-5 acc: 0.9551 ± 0.0028
[Seed:15] KFold-5 acc: 0.9554 ± 0.0055
[Seed:16] KFold-5 acc: 0.9551 ± 0.0030
[Seed:17] KFold-5 acc: 0.9558 ± 0.0036
[Seed:18] KFold-5 acc: 0.9553 ± 0.0010
[Seed:19] KFold-5 acc: 0.9565 ± 0.0024
Fitting 5 folds for each of 20 candidates, totalling 100 fits

✅ [RandomSearch] best_acc: 0.9449
📌 best_params: {'clf__max_depth': None, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 4, 'clf__min_samples_split': 